# pyRadPlan bridge in pyCERR

This notebook demonstrates **`cerr.imrtp.pyradplan_bridge`** — the bridge that
feeds pyCERR geometry into [pyRadPlan](https://github.com/e0404/pyRadPlan) for
photon/proton **dose calculation** and **fluence optimization**, then imports
the resulting dose back into `planC`.

**Pipeline:** `planC` scan/structures/beams → pyRadPlan `CT`/`cst`/`Plan` →
beamlet dose-influence matrix (`dij`) → fluence optimization → `planC.dose`.

**Requires:** `pip install pyRadPlan` (0.4.x) in the `pycerr` environment.

We use a self-contained water-box phantom so the notebook runs with no
external data. Swap in `pc.loadDcmDir(...)` to use your own DICOM (see the
real-world notebook).

## 1. Build a phantom `planC`

In [1]:
import numpy as np
import SimpleITK as sitk
from cerr import plan_container as pc
from cerr.imrtp import pyradplan_bridge as prp

def make_phantom(voxel_mm=4.0, shape_zyx=(28, 44, 44), radius_mm=14.0,
                 tmp_dir=None):
    """Water-box phantom with a BODY box and a central spherical PTV.

    Returns a pyCERR ``planC`` with scan 0 and structures BODY, PTV.
    """
    import tempfile, os
    nz, ny, nx = shape_zyx
    hu = np.full(shape_zyx, -1000.0, dtype=np.float32)   # air
    hu[3:-3, 6:-6, 6:-6] = 0.0                           # water box
    img = sitk.GetImageFromArray(hu)
    img.SetSpacing([voxel_mm] * 3)
    img.SetOrigin([0.0, 0.0, 0.0])
    tmp_dir = tmp_dir or tempfile.mkdtemp()
    nii = os.path.join(tmp_dir, "phantom.nii.gz")
    sitk.WriteImage(img, nii)
    planC = pc.loadNiiScan(nii, imageType="CT SCAN")

    scanImg = planC.scan[0].getSitkImage()
    def _import(mask_zyx, name):
        mImg = sitk.GetImageFromArray(mask_zyx.astype(np.uint8))
        mImg.CopyInformation(scanImg)
        m3 = prp.doseArrayFromSitk(mImg, planC, 0).astype(bool)
        pc.importStructureMask(m3, 0, name, planC)

    body = np.zeros(shape_zyx, dtype=bool)
    body[3:-3, 6:-6, 6:-6] = True
    _import(body, "BODY")
    cz, cy, cx = (nz - 1) / 2, (ny - 1) / 2, (nx - 1) / 2
    zz, yy, xx = np.ogrid[:nz, :ny, :nx]
    dist = voxel_mm * np.sqrt((zz-cz)**2 + (yy-cy)**2 + (xx-cx)**2)
    _import(dist <= radius_mm, "PTV")
    return planC

def struct_index(planC, name):
    return [i for i, s in enumerate(planC.structure)
            if s.structureName == name][0]

planC = make_phantom()
ptv = struct_index(planC, "PTV")
body = struct_index(planC, "BODY")
print("scan size:", tuple(planC.scan[0].getScanSize()),
      "| structures:", [s.structureName for s in planC.structure])

scan size: (np.int64(44), np.int64(44), np.int64(28)) | structures: ['BODY', 'PTV']


## 2. Convert geometry and compute the dose-influence matrix

`planFromPlanC` builds the pyRadPlan `CT`, structure set (`cst`) and `Plan`.
Here we give explicit beam angles; if you omit `gantryAngles` it reads the
treatment beams from `planC.beams[0]` instead.

`calcDoseInfluence` runs pyRadPlan's steering-geometry generation and
pencil-beam engine, returning the steering info (`stf`) and the sparse
beamlet influence matrix (`dij`).

In [2]:
iso = prp.targetCentroidMm(planC, [ptv])          # isocenter = PTV centroid (mm)
ct, cst, pln = prp.planFromPlanC(
    planC, scanNum=0, beamsNum=None,
    objectives={ptv: [prp.squaredDeviation(2.0)],  # PTV -> 2 Gy
                body: [prp.squaredOverdosing(1.0)]},# spare BODY above 1 Gy
    structNums=[ptv, body], targetStructNums=[ptv],
    gantryAngles=[0.0, 90.0, 180.0, 270.0], isoCenter=iso,
    bixelWidth=5.0, prescribedDose=2.0,
    doseGridResolution={"x": 4.0, "y": 4.0, "z": 4.0})

stf, dij = prp.calcDoseInfluence(ct, cst, pln)
A = dij.physical_dose.flat[0] if hasattr(dij.physical_dose, "flat") else dij.physical_dose
print("dose-influence matrix: %d voxels x %d beamlets, nnz=%d"
      % (A.shape[0], A.shape[1], A.nnz))

Beam:   0%|          | 0/4 [00:00<?, ?b/s]

Beam:   0%|          | 0/4 [00:00<?, ?b/s]

C:\Users\aptea\Miniconda3\envs\pycerr\Lib\site-packages\pyRadPlan\raytracer\_siddon.py:359: RuntimeWarning: divide by zero encountered in divide
  (p_min - self._source_points) / self._ray_vec,  # alpha to "near" plane
C:\Users\aptea\Miniconda3\envs\pycerr\Lib\site-packages\pyRadPlan\raytracer\_siddon.py:360: RuntimeWarning: divide by zero encountered in divide
  (p_max - self._source_points) / self._ray_vec,
C:\Users\aptea\Miniconda3\envs\pycerr\Lib\site-packages\numpy\lib\_function_base_impl.py:1496: RuntimeWarning: invalid value encountered in subtract
  a = op(a[slice1], a[slice2])
C:\Users\aptea\Miniconda3\envs\pycerr\Lib\site-packages\pyRadPlan\raytracer\_siddon.py:429: RuntimeWarning: invalid value encountered in multiply
  ijk = sp_scaled[:, :, None] + rv_scaled[:, :, None] * alphas_mid[:, None, :]


Ray:   0%|          | 0/77 [00:00<?, ?r/s]

Ray:  26%|██▌       | 20/77 [00:00<00:00, 186.60r/s]

...

Ray:  31%|███       | 24/77 [00:00<00:00, 214.40r/s]

Ray:  60%|█████▉    | 46/77 [00:00<00:00, 214.35r/s]

Ray:  88%|████████▊ | 68/77 [00:00<00:00, 201.81r/s]

Beam: 100%|██████████| 4/4 [00:03<00:00,  1.11b/s]

dose-influence matrix: 56144 voxels x 308 beamlets, nnz=4333464


## 3. Optimize fluence and import the dose

`optimizeAndImportDose` runs pyRadPlan's fluence optimizer against the
objectives above, resamples the resulting dose onto the CT grid, and appends
it to `planC.dose`.

In [3]:
w, doseNum, planC = prp.optimizeAndImportDose(
    planC, ct, cst, stf, dij, pln, scanNum=0, fractionGroupID="pyRadPlan")
print("optimized %d beamlet weights; dose stored at planC.dose[%d]"
      % (w.size, doseNum))

C:\Users\aptea\Miniconda3\envs\pycerr\Lib\site-packages\pyRadPlan\optimization\problems\_optiprob.py:108: UserWarning: Solver ipopt not available. Choose from ['scipy'], and we will choose the first available one for you!
  warnings.warn(


optimized 308 beamlet weights; dose stored at planC.dose[0]


In [4]:
import matplotlib.pyplot as plt

def show_dose(planC, doseNum, scanNum=0, title="dose"):
    """Overlay a dose colourwash on the central CT slice."""
    scan3M = planC.scan[scanNum].getScanArray()
    dose3M = planC.dose[doseNum].doseArray
    k = scan3M.shape[2] // 2
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(scan3M[:, :, k], cmap="gray")
    m = ax.imshow(np.ma.masked_less(dose3M[:, :, k], 0.05 * dose3M.max()),
                  cmap="jet", alpha=0.5)
    plt.colorbar(m, ax=ax, label="Gy", fraction=0.046)
    ax.set_title("%s (slice %d)  max=%.2f Gy" % (title, k, dose3M.max()))
    ax.axis("off")
    plt.show()

In [5]:
show_dose(planC, doseNum, title="pyRadPlan optimized dose")

C:\Users\aptea\AppData\Local\Temp\ipykernel_13148\1263061914.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Unit-weight (open-field) dose

You can also compute the dose for **unit beamlet weights** (open fields) in one
call straight from the RTPLAN/explicit geometry — useful as a quick forward
dose without optimization.

In [6]:
# calcBeamletDoseAndImport builds the influence matrix and imports the
# unit-weight dose in a single call.
dij2, stf2, doseNum2, planC = prp.calcBeamletDoseAndImport(
    planC, scanNum=0, beamsNum=None if not planC.beams else 0,
    bixelWidth=5.0, structNums=[ptv, body], targetStructNums=[ptv]) \
    if planC.beams else (None, None, None, planC)
print("open-field dose at planC.dose[%s]" % (doseNum2 if planC.beams else "n/a"))

open-field dose at planC.dose[n/a]


## Notes & capabilities

- **Coordinate handling** — pyCERR scans with a flipped `imageOrientationPatient`
  are reoriented to LPS internally so pyRadPlan's ray tracer behaves; the dose is
  resampled back onto the original scan grid on import.
- **Degenerate isocenter** — if `planC.beams` stores a `[0,0,0]` placeholder
  isocenter (common in verification RTPLANs), `planFromPlanC` warns and falls
  back to the target-structure centroid automatically.
- **Protons** — pass beams whose `RadiationType` is `PROTON` (or an `IonPlan`
  radiation mode); the bridge selects pyRadPlan's ion pipeline.
- **Objectives** — `squaredDeviation`, `squaredOverdosing`, `squaredUnderdosing`,
  `meanDose` are thin wrappers so you never import pyRadPlan directly.